# sin_7pi GPU Run — m=7500 and m=10000

GPU-accelerated ODE gradient flow using **PyTorch + torchdiffeq** (dopri5 ≡ RK45).  
Matches `simulate_parallel.py` exactly — same init, tolerances, and output files.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run the **Mount Google Drive** cell first — files save there continuously, so a disconnect can't lose your results

**After it finishes:** Copy `MyDrive/sin7pi_output/sin_7pi/` into your local `figures/Replication data/`, then run `plot_convergence_now.py`.

In [ ]:
# PyTorch is pre-installed and GPU-ready on Colab — only torchdiffeq needed
!pip install torchdiffeq -q

In [ ]:
# ── Mount Google Drive — run this before starting the ODE ────────────────────
# Files are written directly to Drive throughout the run.
# Safe to disconnect: results are already saved when you come back.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/sin7pi_output'
import os; os.makedirs(DRIVE_OUT, exist_ok=True)
print(f'Drive mounted. Output will go to: {DRIVE_OUT}')

In [ ]:
import torch
from torchdiffeq import odeint
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os, csv, time, zipfile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Config — matches simulate_parallel.py exactly ────────────────────────────
TARGET_KEY   = 'sin_7pi'
TARGET_LABEL = r'$\sin(7\pi x)$'
K_TRUE       = 13
M_RUNS       = [7500, 10000]

T_FINAL          = 500
N_QUAD           = 200
N_SAVE           = 300
SEED             = 42
ACTIVE_THRESHOLD = 0.05   # relative: |a_j| > 0.05 * max|a|
GAP_TOL          = 0.02

# Output goes directly to Google Drive — survives disconnects
OUT_BASE = DRIVE_OUT   # set by the Mount Google Drive cell above

INFLECTION_LOCS = np.array([-6/7, -5/7, -4/7, -3/7, -2/7, -1/7, 0.0,
                              1/7,  2/7,  3/7,  4/7,  5/7,  6/7])

# Quadrature grid
x_quad_np = np.linspace(-1.0, 1.0, N_QUAD)
dx_np     = x_quad_np[1] - x_quad_np[0]

# GPU tensors (float64 to match scipy precision)
x_quad     = torch.tensor(x_quad_np, dtype=torch.float64, device=device)
dx         = float(dx_np)
fstar_vals = torch.sin(7 * torch.pi * x_quad)  # on GPU

print('Config OK')

In [ ]:
# ── PyTorch ODE — runs entirely on GPU ───────────────────────────────────────
def make_ode_torch(m):
    """Returns a torchdiffeq-compatible ODE function for sin_7pi, width m."""
    def ode(t, y):
        a        = y[:m]
        b        = y[m:]
        # (N_QUAD, m) ReLU matrix
        relu_mat = torch.clamp(x_quad.unsqueeze(1) - b.unsqueeze(0), min=0.0)
        net      = (a * relu_mat).sum(dim=1)          # (N_QUAD,)
        residual = net - fstar_vals                   # (N_QUAD,)
        da       = -(residual.unsqueeze(1) * relu_mat).sum(0) * dx   # (m,)
        cum_right = torch.cumsum(residual.flip(0), dim=0).flip(0) * dx  # (N_QUAD,)
        idx      = torch.searchsorted(x_quad.contiguous(), b.contiguous()).clamp(0, N_QUAD - 1)
        db       = a * cum_right[idx]                 # (m,)
        return torch.cat([da, db])
    return ode

# ── NumPy helpers — post-processing (CPU, matches simulate_parallel.py) ──────
def network_np(x, a, b):
    return (a * np.maximum(0.0, x[:, None] - b[None, :])).sum(axis=1)

def compute_ode_velocities_np(a, b):
    residual  = network_np(x_quad_np, a, b) - np.sin(7 * np.pi * x_quad_np)
    relu_mat  = np.maximum(0.0, x_quad_np[:, None] - b[None, :])
    da        = -(residual[:, None] * relu_mat).sum(0) * dx_np
    cum_right = np.cumsum(residual[::-1])[::-1] * dx_np
    idx       = np.searchsorted(x_quad_np, b).clip(0, N_QUAD - 1)
    R         = cum_right[idx]
    db        = a * R
    return da, db, R

def count_clusters(biases, tol=GAP_TOL):
    s = np.sort(biases)
    return 1 + int(np.sum(np.diff(s) > tol))

def get_cluster_centers(biases, tol=GAP_TOL):
    s = np.sort(biases)
    centers, group = [], [s[0]]
    for i in range(1, len(s)):
        if s[i] - s[i-1] > tol:
            centers.append(float(np.mean(group)))
            group = []
        group.append(s[i])
    centers.append(float(np.mean(group)))
    return np.array(centers)

print('ODE and helpers defined')

In [ ]:
# ── Figure + CSV writer — identical output format to simulate_parallel.py ────
def save_all_outputs(m, T, out_dir, a_final, b_final,
                     b_traj_sorted, t_save_np, losses_sparse, t_sparse):
    os.makedirs(out_dir, exist_ok=True)
    x_plot = np.linspace(-1.0, 1.0, 500)
    fstar_plot = np.sin(7 * np.pi * x_plot)

    f_final         = network_np(x_plot, a_final, b_final)
    n_clusters      = count_clusters(b_final)
    cluster_centers = get_cluster_centers(b_final)
    da_final, db_final, R_final = compute_ode_velocities_np(a_final, b_final)
    a_max     = max(float(np.abs(a_final).max()), 1e-12)
    is_active = np.abs(a_final) > ACTIVE_THRESHOLD * a_max
    n_active  = int(is_active.sum())
    final_loss = float(0.5 * np.trapz(
        (network_np(x_quad_np, a_final, b_final) - np.sin(7*np.pi*x_quad_np))**2,
        x_quad_np))

    sort_idx = np.argsort(b_final)
    b_s  = b_final[sort_idx];  a_s  = a_final[sort_idx]
    da_s = da_final[sort_idx]; db_s = db_final[sort_idx]
    R_s  = R_final[sort_idx];  active_s = is_active[sort_idx]

    # Figure 1: trajectories / fit / loss
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'target={TARGET_LABEL},  m={m},  T={T}  |  clusters={n_clusters},  k={K_TRUE}', fontsize=12)
    ax = axes[0]
    alpha = min(0.4, 20.0 / m)
    for j in range(m):
        ax.plot(t_save_np, b_traj_sorted[j], color='steelblue', alpha=alpha, linewidth=0.4)
    ax.set_xlabel('Time'); ax.set_ylabel('Bias value')
    ax.set_title('Sorted Bias Trajectories'); ax.set_xlim([0, T])
    ax = axes[1]
    ax.plot(x_plot, fstar_plot, 'k--', lw=2, label=f'Target {TARGET_LABEL}')
    ax.plot(x_plot, f_final, 'r-', lw=2, label='Network $f$')
    y0, y1 = ax.get_ylim()
    tick_h = (y1 - y0) * 0.06
    ax.vlines(cluster_centers, -tick_h/2, tick_h/2, colors='steelblue', linewidths=2.5, zorder=5)
    ax.set_xlabel('$x$'); ax.set_title('Final Fit vs Target')
    ax.legend(fontsize=8); ax.set_xlim([-1, 1])
    ax = axes[2]
    ax.semilogy(t_sparse, losses_sparse, color='darkorange', lw=1.5)
    ax.set_xlabel('Time'); ax.set_ylabel('MSE Loss')
    ax.set_title(f'Loss (log)   final={final_loss:.2e}')
    ax.set_xlim([0, T]); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'slide93_reproduction.png'), bbox_inches='tight'); plt.close()

    # Figure 2: clusters vs inflection points
    x_fine = np.linspace(-1.0, 1.0, 2000)
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.suptitle(f'target={TARGET_LABEL},  m={m},  T={T}  |  clusters={n_clusters},  k={K_TRUE}  (C=k: {n_clusters==K_TRUE})', fontsize=11)
    ax.plot(x_fine, np.sin(7*np.pi*x_fine), 'k-', lw=2, label=f'Target {TARGET_LABEL}')
    ax.plot(x_plot, f_final, 'r-', lw=1.5, label='Final network')
    for cx in cluster_centers: ax.axvline(cx, color='steelblue', alpha=0.5, lw=0.7)
    for ix in INFLECTION_LOCS: ax.axvline(ix, color='green', alpha=0.7, lw=1.2, linestyle='--')
    h, _ = ax.get_legend_handles_labels()
    h += [Line2D([0],[0], color='steelblue', lw=1.5, label=f'Bias clusters ({n_clusters})'),
          Line2D([0],[0], color='green', lw=1.5, ls='--', label=f'Inflection pts k={K_TRUE}')]
    ax.legend(handles=h, fontsize=9); ax.set_xlabel('$x$')
    ax.set_title('Cluster Locations vs Inflection Points')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'clusters_vs_inflections.png'), bbox_inches='tight'); plt.close()

    # Figure 3: ODE stationarity check
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Stationarity — target={TARGET_LABEL},  m={m},  T={T}\nactive={n_active},  k={K_TRUE},  active<=k: {n_active<=K_TRUE}', fontsize=10)
    ax = axes[0]
    ax.scatter(b_s, np.abs(da_s)+1e-15, c=np.abs(a_s), cmap='viridis', s=10)
    ax.set_xlabel('$b_j$'); ax.set_ylabel('$|\\dot{a}_j|$')
    ax.set_title('Amplitude velocity'); ax.set_yscale('log'); ax.grid(True, alpha=0.3)
    ax = axes[1]
    ax.scatter(b_s, np.abs(db_s)+1e-15, c=np.abs(a_s), cmap='viridis', s=10)
    ax.set_xlabel('$b_j$'); ax.set_ylabel('$|\\dot{b}_j|$')
    ax.set_title('Bias velocity'); ax.set_yscale('log'); ax.grid(True, alpha=0.3)
    ax = axes[2]
    sizes = np.clip(np.abs(a_s)/a_max*80, 4, 80)
    sc = ax.scatter(b_s, R_s, c=np.abs(a_s), cmap='viridis', s=sizes)
    ax.scatter(b_s[active_s], R_s[active_s], edgecolors='red', facecolors='none', s=60, lw=1.2, label=f'Active ({n_active})')
    ax.axhline(0.0, color='k', lw=1.0, linestyle='--')
    plt.colorbar(sc, ax=ax, label='$|a_j|$')
    ax.set_xlabel('$b_j$'); ax.set_ylabel('$R_j$')
    ax.set_title(f'Integrated residual  active={n_active} <= k={K_TRUE}: {n_active<=K_TRUE}')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'ode_verification.png'), bbox_inches='tight'); plt.close()

    # Figure 4: final_fit_clean.png
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x_plot, fstar_plot, 'k--', lw=2, label=f'Target {TARGET_LABEL}')
    ax.plot(x_plot, f_final, color='steelblue', lw=2, label='Network $f$')
    y0, y1 = ax.get_ylim()
    ax.vlines(cluster_centers, y0, y0+(y1-y0)*0.07, colors='crimson', linewidths=2.5, zorder=5, label=f'Cluster centers ({n_clusters})')
    ax.set_xlabel('$x$')
    ax.set_title(f'{TARGET_LABEL},  $m={m}$,  $T={T}$  |  $C={n_clusters}$,  $k={K_TRUE}$  (C=k: {n_clusters==K_TRUE})', fontsize=11)
    ax.legend(fontsize=9); ax.set_xlim([-1, 1]); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'final_fit_clean.png'), bbox_inches='tight'); plt.close()

    # convergence_check.csv
    R_tol = 1e-3
    with open(os.path.join(out_dir, 'convergence_check.csv'), 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['neuron_idx','b_j','a_j','da_dt','db_dt','R_j','is_active','R_near_zero'])
        for i in range(m):
            w.writerow([sort_idx[i], f'{b_s[i]:.6f}', f'{a_s[i]:.6f}',
                        f'{da_s[i]:.6e}', f'{db_s[i]:.6e}', f'{R_s[i]:.6e}',
                        int(active_s[i]), int(abs(R_s[i]) < R_tol)])

    # run_meta.csv
    fields = ['target','m','T','k_true','n_clusters','n_active','loss','max_da','max_db','active_leq_k']
    meta = {'target': TARGET_KEY, 'm': m, 'T': T, 'k_true': K_TRUE,
            'n_clusters': n_clusters, 'n_active': n_active, 'loss': final_loss,
            'max_da': float(np.abs(da_final).max()),
            'max_db': float(np.abs(db_final).max()),
            'active_leq_k': int(n_active <= K_TRUE)}
    with open(os.path.join(out_dir, 'run_meta.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader(); w.writerow({k: meta[k] for k in fields})

    return meta

print('Output functions defined')

In [ ]:
# ── Run both m values ─────────────────────────────────────────────────────────
for m in M_RUNS:
    print(f'\n{"="*60}')
    print(f'sin_7pi  m={m}  T={T_FINAL}')
    print(f'{"="*60}')

    out_dir = os.path.join(OUT_BASE, TARGET_KEY, f'm={m}', f'T={T_FINAL}')

    # Initialisation — matches simulate_parallel.py (seed=42)
    rng   = np.random.default_rng(SEED)
    a0_np = rng.standard_normal(m) * 0.01
    b0_np = rng.uniform(-1.0, 1.0, m)
    y0    = torch.tensor(np.concatenate([a0_np, b0_np]),
                         dtype=torch.float64, device=device)

    t_eval = torch.tensor(np.linspace(0.0, T_FINAL, N_SAVE),
                          dtype=torch.float64, device=device)

    print(f'  Solving on {device} ...')
    t_wall = time.time()

    with torch.no_grad():
        ys = odeint(
            make_ode_torch(m),
            y0,
            t_eval,
            method='dopri5',
            rtol=1e-4,
            atol=1e-6,
        )  # shape: (N_SAVE, 2m)

    elapsed = time.time() - t_wall
    print(f'  ODE done in {elapsed:.1f}s  ({elapsed/60:.1f} min)')

    # Move to CPU numpy for all post-processing
    ys_np   = ys.cpu().numpy()       # (N_SAVE, 2m)
    t_np    = t_eval.cpu().numpy()   # (N_SAVE,)
    a_final = ys_np[-1, :m]
    b_final = ys_np[-1, m:]

    # Sorted bias trajectories for trajectory plot — (m, N_SAVE)
    sort_order    = np.argsort(b_final)
    b_traj_sorted = ys_np[:, m:][:, sort_order].T

    # Sparse losses (every 5th save point)
    snap_idx = np.arange(0, N_SAVE, 5)
    losses_sparse = np.array([
        0.5 * np.trapz(
            (network_np(x_quad_np, ys_np[i, :m], ys_np[i, m:])
             - np.sin(7*np.pi*x_quad_np))**2,
            x_quad_np)
        for i in snap_idx
    ])

    meta = save_all_outputs(
        m, T_FINAL, out_dir,
        a_final, b_final,
        b_traj_sorted, t_np,
        losses_sparse, t_np[snap_idx],
    )

    print(f'  C={meta["n_clusters"]}  k={K_TRUE}  C=k: {meta["n_clusters"]==K_TRUE}')
    print(f'  max|da|={meta["max_da"]:.4f}  max|db|={meta["max_db"]:.4f}  loss={meta["loss"]:.3e}')
    print(f'  Saved -> {out_dir}')

In [ ]:
# ── Files are already in Google Drive — nothing to download ──────────────────
# Results were written directly to Drive throughout the run.
# Even if Colab disconnected, your files are safe.

print('Run complete. Files saved to Google Drive:')
print(f'  {DRIVE_OUT}/')
print()
print('To use locally:')
print('  1. Open Google Drive in your browser')
print('  2. Navigate to MyDrive/sin7pi_output/')
print('  3. Download the sin_7pi/ folder (right-click → Download)')
print('  4. Paste sin_7pi/ into:  figures/Replication data/')
print('  5. Run locally:  python plot_convergence_now.py')

# Confirm files exist
import os
for root, dirs, files in os.walk(DRIVE_OUT):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'  ✓  {os.path.relpath(path, DRIVE_OUT)}  ({size/1024:.1f} KB)')